In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# ***Problem Statement***
***Predict the daily number of audience members visiting each movie theater using historical booking and visit records, theater metadata, and calendar features. The goal is to produce accurate day-level audience forecasts for every theater in the sample_submission.csv file.***

# ***Importing Libraries***

In [ ]:
import numpy as np
import pandas as pd

from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score, make_scorer

# Machine Learning Models
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

print('Libraries loaded')

# ***Data Loading***

In [ ]:
booknow_booking = pd.read_csv("/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv")
booknow_theaters = pd.read_csv("/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_theaters/booknow_theaters.csv")
booknow_visits = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_visits/booknow_visits.csv')
cinePOS_booking = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_booking/cinePOS_booking.csv')
cinePOS_theaters = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_theaters/cinePOS_theaters.csv')
date_info = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/date_info/date_info.csv')
movie_theater_id_relation = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/movie_theater_id_relation/movie_theater_id_relation.csv')
sample_submission = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv')

In [ ]:
print(f'booknow_booking shape: {booknow_booking.shape}')
print(f'booknow_theaters shape: {booknow_theaters.shape}')
print(f'booknow_visits shape: {booknow_visits.shape}')
print(f'cinePOS_booking shape: {cinePOS_booking.shape}')
print(f'cinePOS_theaters shape: {cinePOS_theaters.shape}')
print(f'date_info shape: {date_info.shape}')
print(f'movie_theater_id_relation shape: {movie_theater_id_relation.shape}')
print(f'sample_submission shape: {sample_submission.shape}')

In [ ]:
booknow_booking.head(2)
# booknow_booking['book_theater_id'].unique()

In [ ]:
booknow_theaters.head(2)
# booknow_theaters['book_theater_id'].unique()

In [ ]:
booknow_visits.head()
# booknow_visits['book_theater_id'].unique()

In [ ]:
cinePOS_booking.head(2)

In [ ]:
cinePOS_theaters.head(2)

In [ ]:
date_info.head()

In [ ]:
movie_theater_id_relation.head(2)

In [ ]:
sample_submission.head(2)

## ***Train Data***

In [ ]:
train_data = booknow_visits.copy()
train_data.rename(columns={'show_date': 'date'}, inplace=True)
train_data['date'] = pd.to_datetime(train_data['date'])
print(f'Shape: {train_data.shape}')
train_data.head()

## ***Test Data***

In [ ]:
test_data = sample_submission.copy()
test_data['book_theater_id'] = test_data['ID'].apply(lambda x: '_'.join(x.split('_')[:2]))
test_data['date'] = test_data['ID'].apply(lambda x: x.split('_')[2])
test_data['date'] = pd.to_datetime(test_data['date'])
print(f'Shape: {test_data.shape}')
test_data.head()

# ***Exploratory Data Analysis(EDA)***

In [ ]:
train_data.shape, test_data.shape

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

## **Categorical Columns in dataset**

In [ ]:
train_data.select_dtypes(include=['object', 'category', 'string']).columns

## **Numerical Columns in dataset**

In [ ]:
train_data.select_dtypes(include=['number']).columns

In [ ]:
test_data.info()

In [ ]:
train_data.isnull().sum()

In [ ]:
plt.figure(figsize=(8,8))
sns.histplot(train_data['audience_count'], kde=True)
plt.title("Distribution of Audience Count")
plt.xlabel("audience_count")
plt.show()

In [ ]:
daily = train_data.groupby('date')['audience_count'].sum()

plt.figure(figsize=(12,4))
plt.plot(daily.index, daily.values)
plt.title("Audience Trend Over Time")
plt.xlabel("Date")
plt.ylabel("Total Audience Count")
plt.show()


In [ ]:
train_data['weekday'] = train_data['date'].dt.weekday

weekday_avg = train_data.groupby('weekday')['audience_count'].mean()

plt.figure(figsize=(7,5))
weekday_avg.plot(kind='bar')
plt.title("Average Audience by Weekday")
plt.xlabel("Weekday (0=Mon ... 6=Sun)")
plt.ylabel("Average Audience")
plt.show()


In [ ]:
train_data['month'] = train_data['date'].dt.month

month_avg = train_data.groupby('month')['audience_count'].mean()

plt.figure(figsize=(7,5))
month_avg.plot(kind='bar')
plt.title("Average Audience by Month")
plt.xlabel("Month")
plt.ylabel("Average Audience")
plt.show()


In [ ]:
theater_totals = train_data.groupby('book_theater_id')['audience_count'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
theater_totals.plot(kind='bar')
plt.title("Top 10 Theaters by Total Audience")
plt.ylabel("Total Audience")
plt.show()

# **Feature Engineering**

In [ ]:
# Renaming show_date → date
date_info.rename(columns={'show_date': 'date'}, inplace=True)

# Converting to datetime format
date_info['date'] = pd.to_datetime(date_info['date'])

# Merge into train and test
train_data = train_data.merge(date_info, on='date', how='left')
test_data = test_data.merge(date_info, on='date', how='left')

# Check first rows
train_data.head()


In [ ]:
test_data.shape

In [ ]:
def add_date_feats(df):
    df['day'] = df['date'].dt.day
    df['weekday'] = df['date'].dt.weekday            # Mon=0 .. Sun=6
    df['is_weekend'] = df['weekday'].isin([5,6]).astype(int)
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['date'].dt.quarter
    return df

train_data = add_date_feats(train_data)
test_data  = add_date_feats(test_data)

In [ ]:
grp = train_data.groupby('book_theater_id')['audience_count']
theater_mean = grp.mean().rename('theater_mean')
theater_median = grp.median().rename('theater_median')
theater_std = grp.std().fillna(0).rename('theater_std')
theater_count = grp.count().rename('theater_count')

agg_df = pd.concat([theater_mean, theater_median, theater_std, theater_count], axis=1).reset_index()
agg_df.head()

# merge into train and test
train_data = train_data.merge(agg_df, on='book_theater_id', how='left')
test_data  = test_data.merge(agg_df, on='book_theater_id', how='left')

# fill missing (theaters in test not in train)
for c in ['theater_mean','theater_median','theater_std','theater_count']:
    test_data[c].fillna(train_data['audience_count'].mean(), inplace=True)
    train_data[c].fillna(train_data['audience_count'].mean(), inplace=True)


In [ ]:
# train_data = train_data.sort_values(['book_theater_id','date']).reset_index(drop=True)

# # adding lag and roll
# def add_lags_rolls(df, lags=[1,7], windows=[3,7]):
#     df = df.copy()
#     df[['lag_'+str(l) for l in lags]] = np.nan  # placeholder
#     for l in lags:
#         df['lag_'+str(l)] = df.groupby('book_theater_id')['audience_count'].shift(l)
  
#     for w in windows:
#         df[f'roll_mean_{w}'] = df.groupby('book_theater_id')['audience_count'].shift(1).rolling(window=w, min_periods=1).mean().reset_index(level=0, drop=True)
#     return df

# train_data = add_lags_rolls(train_data, lags=[1,7], windows=[3,7])

# train_data[['lag_1','lag_7','roll_mean_3','roll_mean_7']].head(10)

In [ ]:
train_data.isnull().sum()

In [ ]:
# train_data = train_data.dropna(subset=['lag_1','lag_7','roll_mean_3','roll_mean_7'])
# train_data.isnull().sum()

# Label Encoding

In [ ]:
le = LabelEncoder()
all_ids = pd.concat([train_data['book_theater_id'], test_data['book_theater_id']]).astype(str)
le.fit(all_ids)
train_data['book_id_le'] = le.transform(train_data['book_theater_id'].astype(str))
test_data['book_id_le']  = le.transform(test_data['book_theater_id'].astype(str))


In [ ]:
# dropping day of week column
train_data.drop('day_of_week', axis = 1, inplace = True)
test_data.drop('day_of_week', axis = 1, inplace = True)

In [ ]:
train_data.head(2)

In [ ]:
test_data.head(2)

# **Train / Valid Split**

In [ ]:
train_data['date'].min(), train_data['date'].max()

In [ ]:
train = train_data[train_data['date'] < "2024-01-01"]
valid = train_data[train_data['date'] >= "2024-01-01"]

In [ ]:
X_train = train.drop(['audience_count','book_theater_id','date'], axis=1)
y_train = train['audience_count']

X_valid = valid.drop(['audience_count','book_theater_id','date'], axis=1)
y_valid = valid['audience_count']

## **X_train & y_train shape**

In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_train.head(2)

In [ ]:
y_train.head(2)

## **X_valid & y_valid shape**

In [ ]:
X_valid.shape, y_valid.shape

In [ ]:
X_valid.head(2)

In [ ]:
y_valid.head(2)

# **Preparing test data for model**

In [ ]:
# full = pd.concat([train_data, test_data], axis=0)
# full = full.sort_values(["book_theater_id", "date"])
# full["lag_1"] = full.groupby("book_theater_id")["audience_count"].shift(1)
# full["lag_7"] = full.groupby("book_theater_id")["audience_count"].shift(7)

# full["roll_mean_3"] = full.groupby("book_theater_id")["audience_count"].transform(lambda x: x.rolling(3).mean())
# full["roll_mean_7"] = full.groupby("book_theater_id")["audience_count"].transform(lambda x: x.rolling(7).mean())

In [ ]:
# Separating train data and test data

# train_processed = full[full["audience_count"] != 0]
# test_processed  = full[full["audience_count"] == 0]
# train_processed.head(2)


In [ ]:
X_test = test_data.drop(["ID", "audience_count","date", "book_theater_id"], axis = 1)
print(X_test.shape)

# ***Model Building***

## **Random Forest**

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_valid)
# rf_mae = mean_absolute_error(y_valid, rf_preds)
# print("Random Forest MAE:", rf_mae)
r2_score(y_valid, rf_preds)

## **XGBoost**

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_valid)
r2_score(y_valid, xgb_preds)

## **CatBoost**

In [ ]:
from catboost import CatBoostRegressor, Pool

model = CatBoostRegressor(
    loss_function='MAE',
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    verbose=200,
    task_type="CPU"
)

model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

preds = model.predict(X_valid)
mae = mean_absolute_error(y_valid, preds)
r2 = r2_score(y_valid, preds)

print("MAE:", mae)
print("R2 Score:", r2)

# **HyperParameter Tuning**

In [ ]:
#XGBoost
xgb_model = XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42)

xgb_param = {
    'max_depth': [2, 4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.08, 0.1],
    'n_estimators': [200, 400, 800]
}

#Catboost

cat_model = CatBoostRegressor(loss_function="MAE",task_type='CPU',random_state=42,verbose=False)

cat_param = {
    'depth': [4, 6, 8],
    'learning_rate': [0.02, 0.03, 0.05],
    'l2_leaf_reg': [3, 5, 7],
    'iterations': [500, 800, 1000]
}


tscv = TimeSeriesSplit(n_splits=3)
scorer = 'neg_mean_absolute_error'

In [ ]:
# XGBoost tuning
xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param,
    n_iter=20,
    scoring=scorer,
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)
xgb_search.fit(X_train, y_train)

print("Best XGB params:", xgb_search.best_params_)
print("Best XGB (neg MAE):", xgb_search.best_score_)

In [ ]:
#CatBoost tuning

cat_search = RandomizedSearchCV(
    estimator=cat_model,
    param_distributions=cat_param,
    n_iter=10,
    scoring=scorer,
    cv=3,
    random_state=42,
    verbose=2
)

cat_search.fit(X_train, y_train)

print("Best params:", cat_search.best_params_)

In [ ]:
best_cat = CatBoostRegressor(
    loss_function='MAE',
    random_state=42,
    verbose=100,
    learning_rate= 0.03,
    l2_leaf_reg= 7,
    iterations=800,
    depth = 8   
)

best_cat.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)

cat_val_pred = best_cat.predict(X_valid)

print("CAT MAE on valid:", mean_absolute_error(y_valid, cat_val_pred))
print("CAT R2 on valid:", r2_score(y_valid, cat_val_pred))

In [ ]:

best_xgb = xgb_search.best_estimator_

xgb_val_pred = best_xgb.predict(X_valid)

print("XGB MAE on valid:", mean_absolute_error(y_valid, xgb_val_pred))
print("XGB R2 on valid:", r2_score(y_valid, xgb_val_pred))

In [ ]:
X_test.shape

In [ ]:
predictions = best_cat.predict(X_test)
predictions

# **Final Submission**

In [ ]:
sub = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv')
sub['audience_count'] = predictions
submission_final = sub[['ID','audience_count']]
submission_final.to_csv('submission.csv', index=False)